# Bank Customer Churn Prediction & Retention Analytics
**Internship submission notebook — Vaishnavi**

Workflow: data understanding → quality checks → EDA → leakage-safe models → validation-based threshold tuning → untouched test evaluation → business recommendations.

**Methodology:** Model comparison is descriptive; Gradient Boosting is the pre-specified retention-focused candidate for threshold tuning. Never choose a threshold using test-set results.

## 1. Load dataset

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             fbeta_score, roc_auc_score, confusion_matrix,
                             ConfusionMatrixDisplay)
import joblib

DATA_PATH = Path('Churn_Modelling.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path.cwd() / 'Bank_Churn_Threshold_Tuning' / 'Churn_Modelling.csv'
if not DATA_PATH.exists():
    raise FileNotFoundError('Keep Churn_Modelling.csv in the notebook working directory.')
df = pd.read_csv(DATA_PATH)
print('Dataset shape:', df.shape)
print('Churn rate:', round(df['Exited'].mean(), 4))

## 2. Data quality and data dictionary

In [ ]:
print('Shape:', df.shape)
print('Duplicate rows:', df.duplicated().sum())
print('Missing values by column:')
display(df.isna().sum().to_frame('Missing'))
display(pd.DataFrame({'Type':df.dtypes.astype(str),'Unique':df.nunique(),'Missing':df.isna().sum()}))
display(df.describe(include='all').T)

## 3. Exploratory data analysis
Churn relationships are descriptive, **not proof of causation**.

In [ ]:
import seaborn as sns
from pathlib import Path
Path('outputs').mkdir(exist_ok=True)
print('Overall churn rate:',round(df['Exited'].mean()*100,2),'%')
for col in ['Geography','Gender','IsActiveMember','NumOfProducts']:
    print('\n',col)
    display(df.groupby(col)['Exited'].agg(['size','mean']).rename(columns={'size':'Customers','mean':'Churn rate'}))
fig, axes=plt.subplots(1,3,figsize=(15,4))
sns.countplot(data=df,x='Exited',ax=axes[0]); axes[0].set_title('Churn distribution')
sns.barplot(data=df,x='Geography',y='Exited',ax=axes[1],errorbar=None);axes[1].set_title('Churn by geography')
sns.boxplot(data=df,x='Exited',y='Age',ax=axes[2]);axes[2].set_title('Age by churn status')
plt.tight_layout();plt.savefig('outputs/eda_churn.png',dpi=140);plt.show()

## 4. Leakage-safe train / validation / test split
The split is 60% training, 20% validation and 20% untouched test. Preprocessing is fitted on training data only.

In [ ]:
X = df.drop(columns=['Exited', 'RowNumber', 'CustomerId', 'Surname'])
y = df['Exited'].astype(int)

# Hold out test before threshold selection. Split remainder into train and validation.
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_dev, y_dev, test_size=0.25, stratify=y_dev, random_state=42)
print('Train:', len(X_train), 'Validation:', len(X_val), 'Test:', len(X_test))

num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = X.select_dtypes(exclude='number').columns.tolist()
preprocessor = ColumnTransformer([
    ('numeric', Pipeline([('imputer', SimpleImputer(strategy='median')),
                          ('scaler', StandardScaler())]), num_cols),
    ('categorical', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                              ('encoder', OneHotEncoder(handle_unknown='ignore'))]), cat_cols)
])
model = Pipeline([
    ('preprocess', preprocessor),
    ('classifier', HistGradientBoostingClassifier(max_iter=120, random_state=42))
])
model.fit(X_train, y_train)
val_prob = model.predict_proba(X_val)[:, 1]

## 5. Compare five model families on validation data
All pipelines share the same training split. Validation results are exploratory; the final test set is reserved for the pre-specified Gradient Boosting threshold comparison.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score
from sklearn.base import clone
models={
 'Logistic Regression': LogisticRegression(max_iter=1000,random_state=42),
 'Decision Tree': DecisionTreeClassifier(max_depth=6,random_state=42),
 'Random Forest': RandomForestClassifier(n_estimators=200,min_samples_leaf=3,random_state=42,n_jobs=-1),
 'Gradient Boosting': HistGradientBoostingClassifier(max_iter=120,random_state=42),
 'Neural Network (MLP)': MLPClassifier(hidden_layer_sizes=(32,16),max_iter=300,early_stopping=True,random_state=42)
}
records=[]
for name,estimator in models.items():
    candidate=Pipeline([('preprocess',clone(preprocessor)),('classifier',estimator)])
    candidate.fit(X_train,y_train)
    prob=candidate.predict_proba(X_val)[:,1]
    pred=(prob>=0.5).astype(int)
    records.append({'Model':name,'Accuracy':accuracy_score(y_val,pred),
                    'Precision':precision_score(y_val,pred,zero_division=0),
                    'Recall':recall_score(y_val,pred,zero_division=0),
                    'F1':f1_score(y_val,pred,zero_division=0),
                    'ROC_AUC':roc_auc_score(y_val,prob)})
model_comparison=pd.DataFrame(records).sort_values('F1',ascending=False)
display(model_comparison)
model_comparison.to_csv('outputs/model_comparison_validation.csv',index=False)

## 6. Tune the pre-specified Gradient Boosting model using validation data only
Validation thresholds are chosen by F1 among thresholds meeting **illustrative** minimum recall 0.55 and accuracy 0.82; otherwise highest F1 overall.

In [ ]:
# Balanced selection: tune on validation ONLY. Do not select using test metrics.
thresholds = np.round(np.arange(0.15, 0.651, 0.025), 3)
thresholds = np.unique(np.append(thresholds, 0.50))
rows = []
for threshold in thresholds:
    val_pred = (val_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, val_pred, labels=[0, 1]).ravel()
    rows.append({
        'Threshold': float(threshold),
        'Accuracy': accuracy_score(y_val, val_pred),
        'Precision': precision_score(y_val, val_pred, zero_division=0),
        'Recall': recall_score(y_val, val_pred, zero_division=0),
        'F1': f1_score(y_val, val_pred, zero_division=0),
        'TN': int(tn), 'FP': int(fp), 'FN': int(fn), 'TP': int(tp)
    })
validation_scores = pd.DataFrame(rows)
min_recall, min_accuracy = 0.55, 0.82  # illustrative goals; adjust ONLY before inspecting test results
eligible = validation_scores[(validation_scores['Recall'] >= min_recall) &
                             (validation_scores['Accuracy'] >= min_accuracy)]
if eligible.empty:
    print('No threshold meets both validation constraints; falling back to maximum validation F1.')
    eligible = validation_scores
else:
    print('Selecting maximum validation F1 among thresholds satisfying both constraints.')
selected = eligible.sort_values(['F1', 'Threshold'], ascending=[False, False]).iloc[0]
chosen_threshold = float(selected['Threshold'])
print('Chosen threshold (validation only):', chosen_threshold)
print('Selected validation metrics:')
print(selected.to_string())
print('Validation baseline at 0.50:')
print(validation_scores[np.isclose(validation_scores['Threshold'], 0.50)].to_string(index=False))
validation_scores.to_csv('outputs/balanced_validation_results.csv', index=False)
display(validation_scores.sort_values('F1', ascending=False).head(10))


## 7. Final test evaluation (do not reselect threshold after viewing these results)

In [ ]:
# Evaluate selected threshold ONCE on untouched test data.
test_prob = model.predict_proba(X_test)[:, 1]
def evaluate(threshold):
    pred = (test_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0, 1]).ravel()
    return {
        'Threshold': threshold,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred, zero_division=0),
        'Recall': recall_score(y_test, pred, zero_division=0),
        'F1': f1_score(y_test, pred, zero_division=0),
        'ROC_AUC': roc_auc_score(y_test, test_prob),
        'TN': int(tn), 'FP': int(fp), 'FN': int(fn), 'TP': int(tp)
    }
final_scores = pd.DataFrame([evaluate(0.50), evaluate(chosen_threshold)],
                            index=['Default 0.50', 'Balanced validation-tuned'])
print(final_scores.round(4).to_string())
final_scores.to_csv('outputs/balanced_final_test_comparison.csv')
joblib.dump({'model': model, 'threshold': chosen_threshold,
             'features': X.columns.tolist()}, 'outputs/balanced_churn_model.joblib')
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for i, threshold in enumerate([0.50, chosen_threshold]):
    ConfusionMatrixDisplay.from_predictions(y_test, (test_prob >= threshold).astype(int),
                                            labels=[0, 1], cmap='Blues', ax=ax[i], colorbar=False)
    ax[i].set_title(f'Test threshold = {threshold:.3f}')
plt.tight_layout()
plt.savefig('outputs/balanced_confusion_matrices.png', dpi=150)
plt.show()
print('Saved: balanced_validation_results.csv, balanced_final_test_comparison.csv, balanced_confusion_matrices.png, balanced_churn_model.joblib')


## 8. Business insights, limitations and recommendations
- Inactive customers and customers in Germany have higher **observed** churn rates in this dataset; these are associations, not causal findings.
- A lower threshold detects more actual churners but increases false positives and outreach costs.
- Prioritize review of high-risk customers, then test retention actions with a controlled experiment before claiming impact.
- Dataset has no transaction history, campaign costs or intervention outcomes; do **not** label balance differences as withdrawals or claim measured retention ROI.
- Risk probabilities and cutoffs require calibration and business validation before operational use.

**Reproducibility:** Save and report the actual final test metrics produced above. The notebook intentionally keeps the validation/test distinction explicit.